In [ ]:
#  - id: uuid nullable=NO max_len=None
#   - created_at: timestamp without time zone nullable=NO max_len=None
#   - updatedAt: timestamp without time zone nullable=NO max_len=None
#   - deletedAt: timestamp without time zone nullable=YES max_len=None
#   - name: text nullable=NO max_len=None
#   - lat: double precision nullable=NO max_len=None
#   - address: text nullable=YES max_len=None
#   - poi_type: text nullable=YES max_len=None
#   - total_reviews: integer nullable=YES max_len=None
#   - lon: double precision nullable=NO max_len=None
#   - geom: USER-DEFINED nullable=NO max_len=None
#   - stay_time: double precision nullable=YES max_len=None
#   - avg_stars: double precision nullable=YES max_len=None
#   - normalize_stars_reviews: double precision nullable=YES max_len=None
#   - open_hours: json nullable=YES max_len=None
#   - poi_type_clean: text nullable=YES max_len=None
#   - main_subcategory: text nullable=YES max_len=None
#   - specialization: text nullable=YES max_len=None
#   - travel_type: json nullable=YES max_len=None

In [1]:
import os
import sys
import csv
import argparse
import psycopg2
import pandas as pd
from psycopg2 import sql
from dotenv import load_dotenv
import csv
from psycopg2.extras import RealDictCursor
import json
import ast
# Load .env from project root
load_dotenv()

True

In [2]:
# Env keys (fallbacks)
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = int(os.getenv("DB_PORT", "5432"))
DB_NAME = os.getenv("DB_NAME", "postgres")
DB_USER = os.getenv("DB_USER", "")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
CONNECT_TIMEOUT = int(os.getenv("DB_CONNECT_TIMEOUT", "10"))

In [3]:
def get_db_connection():
    """
    Trả về psycopg2 connection sử dụng biến môi trường từ .env.
    """
    conn_kwargs = {
        "host": DB_HOST,
        "port": DB_PORT,
        "dbname": DB_NAME,
        "user": DB_USER,
        "password": DB_PASSWORD,
        "connect_timeout": CONNECT_TIMEOUT,
    }
    return psycopg2.connect(cursor_factory=RealDictCursor, **conn_kwargs)

def test_connection(sql_query: str = "SELECT now() AS now"):
    """
    Thực thi 1 câu SQL test và in kết quả.
    """
    conn = None
    try:
        conn = get_db_connection()
        with conn.cursor() as cur:
            cur.execute(sql_query)
            rows = cur.fetchall()
            print(f"✓ Query executed: {sql_query}")
            for row in rows:
                print(row)
    except Exception as e:
        print(f"DB error: {e}")
    finally:
        if conn:
            conn.close()

def list_tables():
    """Liệt kê các bảng hiện có (bỏ schema hệ thống)."""
    sql_query = """
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_type='BASE TABLE'
      AND table_schema NOT IN ('pg_catalog','information_schema')
    ORDER BY table_schema, table_name;
    """
    test_connection(sql_query)

In [4]:
list_tables()

✓ Query executed: 
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_type='BASE TABLE'
      AND table_schema NOT IN ('pg_catalog','information_schema')
    ORDER BY table_schema, table_name;
    
RealDictRow([('table_schema', 'public'), ('table_name', 'PoiClean')])
RealDictRow([('table_schema', 'public'), ('table_name', 'spatial_ref_sys')])


In [5]:
# Mở connection
conn = get_db_connection()
cur = conn.cursor()

In [19]:
# cur.execute(
#     'SELECT * FROM "PoiClean" WHERE id = %s',
#     ("c49cdc27-ea21-42fc-a857-d2488b0a7ba7",)
# )

# rows = cur.fetchall()

# for poi in rows:
#     print(poi.get('poi_type_clean'))


In [14]:
cur.execute('SELECT * FROM "PoiClean"')
rows = cur.fetchall()
for poi in rows:
    print(poi.get('poi_type_clean'))

Bar
Restaurant
Bar
Bar
Bar
Restaurant
Restaurant
Market
Flea market
Bar
Bar
Restaurant
Tour operator
Supermarket
Rock climbing gym
Beach
Army museum
Restaurant
Restaurant
Castle
War museum
Catholic church
Buddhist temple
Restaurant
Historical landmark
Market
Buddhist temple
Park
Tourist attraction
Catholic church
Hotel
Historical landmark
Restaurant
Park
Beach volleyball court
Historical landmark
Restaurant
City park
Beach clothing store
Restaurant
Supermarket
Bar
Supermarket
Market
Restaurant
Restaurant
Wholesale plant nursery
Restaurant
Restaurant
Restaurant
Restaurant
Bar
Bar
Historical landmark
Restaurant
Restaurant
Restaurant
Bar
Bar
Market
Historical landmark
Bar
Bar
Bar
Restaurant
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Castle
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Cafe
Museum
Cafe
Cafe
Cafe
Cafe
Cafe

# import csv - open_hours

In [7]:
file_path = os.path.join(os.getcwd(), "data/new_data_final_filled.csv")
df = pd.read_csv(file_path, encoding="utf-8")
df.head(1)

,id,name,address,lat,lon,poi_type,avg_stars,total_reviews,stay_time,normalize_stars_reviews,crowd,offerings,atmosphere,highlights,dining_options,children,accessibility,popular_for,opening_hours
0,0f9d2009-9436-46a4-b354-b0261898a39e,The Pub Coffee - Beer & Cocktail,"18A17 Tăng Nhơn Phú, Phước Long B, Quận 9, Thà...",10.829481,106.773785,"Cafe,Bar",4.9,181,30,0.755,Groups,"Alcohol, Beer, Cocktails, Coffee, Hard liquor","Casual, Cozy","Great beer selection, Great coffee, Live music...",Table service,NaN,NaN,NaN,"[{'day': 'Monday', 'hours': [{'start': '00:00'..."


In [8]:
import json
import pandas as pd

# Read the CSV file with opening_hours data
file_path = os.path.join(os.getcwd(), "data/new_data_final_filled.csv")
df_hours = pd.read_csv(file_path, encoding="utf-8", usecols=['id', 'opening_hours'])

# Display first few rows to verify
print(f"Total rows in CSV: {len(df_hours)}")
print(df_hours.head())

Total rows in CSV: 1454
                                     id  \
0  0f9d2009-9436-46a4-b354-b0261898a39e   
1  02887955-963a-43ac-b0f7-355d7d7cfacf   
2  622c7643-30e8-4402-9b6c-b8407ff063e2   
3  4f06908d-e9fa-4f6a-b1ae-c7d8882e2edf   
4  279dfce3-c227-4b58-b4ed-09197327a32a   

                                       opening_hours  
0  [{'day': 'Monday', 'hours': [{'start': '00:00'...  
1  [{'day': 'Monday', 'hours': [{'start': '08:00'...  
2  [{'day': 'Tuesday', 'hours': [{'start': '11:00...  
3  [{'day': 'Tuesday', 'hours': [{'start': '17:30...  
4  [{'day': 'Monday', 'hours': [{'start': '10:00'...  


In [12]:
# Update open_hours in database from CSV data
updated_count = 0
skipped_count = 0
error_count = 0

for idx, row in df_hours.iterrows():
    poi_id = row['id']
    opening_hours = row['opening_hours']
    
    # Skip if opening_hours is null or empty
    if pd.isna(opening_hours) or opening_hours == '':
        skipped_count += 1
        continue
    
    try:
        # convert Python-like string -> valid JSON
        opening_hours_json = json.dumps(ast.literal_eval(opening_hours))

        cur.execute(
            'UPDATE "PoiClean" SET open_hours = %s WHERE id = %s',
            (opening_hours_json, poi_id)
        )
        updated_count += 1
        
        # Print progress every 100 records
        if (updated_count + skipped_count) % 100 == 0:
            print(f"Processed: {updated_count + skipped_count} records (Updated: {updated_count}, Skipped: {skipped_count})")
    except Exception as e:
        error_count += 1
        print(f"Error updating POI {poi_id}: {e}")

# Commit the changes
conn.commit()
print(f"\n✓ Update complete!")
print(f"Total Updated: {updated_count}")
print(f"Total Skipped: {skipped_count}")
print(f"Total Errors: {error_count}")

Processed: 100 records (Updated: 100, Skipped: 0)
Processed: 200 records (Updated: 200, Skipped: 0)
Processed: 300 records (Updated: 300, Skipped: 0)
Processed: 400 records (Updated: 400, Skipped: 0)
Processed: 500 records (Updated: 500, Skipped: 0)
Processed: 600 records (Updated: 600, Skipped: 0)
Processed: 700 records (Updated: 700, Skipped: 0)
Processed: 800 records (Updated: 800, Skipped: 0)
Processed: 900 records (Updated: 900, Skipped: 0)
Processed: 1000 records (Updated: 1000, Skipped: 0)
Processed: 1100 records (Updated: 1100, Skipped: 0)
Processed: 1200 records (Updated: 1200, Skipped: 0)
Processed: 1300 records (Updated: 1300, Skipped: 0)
Processed: 1400 records (Updated: 1400, Skipped: 0)

✓ Update complete!
Total Updated: 1454
Total Skipped: 0
Total Errors: 0


In [13]:
# Verify the update by checking a few POIs
verify_query = 'SELECT id, name, open_hours FROM "PoiClean" WHERE open_hours IS NOT NULL LIMIT 5'
cur.execute(verify_query)
results = cur.fetchall()

print("Sample of updated records:")
for poi in results:
    print(f"\nPOI: {poi['name']} (ID: {poi['id']})")
    print(f"Opening Hours: {poi['open_hours'][:100]}...")  # Print first 100 chars

Sample of updated records:

POI: El City Market (ID: 0f35c945-8e0f-4d86-8795-49df2bd08ed6)
Opening Hours: [{'day': 'Saturday', 'hours': [{'start': '10:00', 'end': '22:00'}]}]...

POI: The Pub Coffee - Beer & Cocktail (ID: 0f9d2009-9436-46a4-b354-b0261898a39e)
Opening Hours: [{'day': 'Monday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Tuesday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Wednesday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Thursday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Friday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Saturday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Sunday', 'hours': [{'start': '00:00', 'end': '23:59'}]}]...

POI: Julieta (ID: 02887955-963a-43ac-b0f7-355d7d7cfacf)
Opening Hours: [{'day': 'Monday', 'hours': [{'start': '08:00', 'end': '20:30'}]}, {'day': 'Tuesday', 'hours': [{'start': '08:00', 'end': '20:30'}]}, {'day': 'Wednesday', 'hours': [{'start': '08:

In [ ]:
# Get statistics on open_hours update
stats_query = '''
SELECT 
    COUNT(*) as total_records,
    SUM(CASE WHEN open_hours IS NOT NULL THEN 1 ELSE 0 END) as with_hours,
    SUM(CASE WHEN open_hours IS NULL THEN 1 ELSE 0 END) as without_hours
FROM "PoiClean"
'''

cur.execute(stats_query)
stats = cur.fetchone()

print("📊 Database Statistics:")
print(f"Total POI records: {stats['total_records']}")
print(f"Records with open_hours: {stats['with_hours']}")
print(f"Records without open_hours: {stats['without_hours']}")
print(f"Coverage: {(stats['with_hours']/stats['total_records']*100):.2f}%")

# import json

In [11]:
import json

In [12]:
file_path = os.path.join(os.getcwd(), "data/results.json")

In [13]:
# Đọc file JSON
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

In [21]:
print(type(data))
print(f"Total POI in JSON: {len(data)}")
print(data[3].get("main_subcategory"))  # In ra 2 POI đầu tiên để kiểm tra

<class 'list'>
Total POI in JSON: 1454
None


In [ ]:
ids = []
for poi in data:
    poi_id = poi['id']
    ids.append(poi_id)
print(len(ids))

1454


In [ ]:
# "poi_type_new": "Bar",
# "main_subcategory": "Live Music Venue",
# "specialization": "Live Entertainment",

In [22]:
# Update fields in database
updated_count = 0
skipped_count = 0
error_count = 0

for poi in data:
    poi_id = poi.get('id')
    poi_type_clean = poi.get('poi_type_new')        # có thể None
    main_subcategory = poi.get('main_subcategory')  # có thể None
    specialization = poi.get('specialization')      # có thể None

    try:
        cur.execute(
            '''UPDATE "PoiClean"
               SET poi_type_clean = %s,
                   main_subcategory = %s,
                   specialization = %s
               WHERE id = %s''',
            (poi_type_clean, main_subcategory, specialization, poi_id)
        )

        updated_count += 1

        # Print progress every 100 records
        if (updated_count + skipped_count) % 100 == 0:
            print(f"Processed: {updated_count + skipped_count} records "
                  f"(Updated: {updated_count}, Skipped: {skipped_count})")

    except Exception as e:
        error_count += 1
        print(f"Error updating POI {poi_id}: {e}")

# Commit
conn.commit()

print("\n✓ Update complete!")
print(f"Total Updated: {updated_count}")
print(f"Total Skipped: {skipped_count}")
print(f"Total Errors: {error_count}")

Processed: 100 records (Updated: 100, Skipped: 0)
Processed: 200 records (Updated: 200, Skipped: 0)
Processed: 300 records (Updated: 300, Skipped: 0)
Processed: 400 records (Updated: 400, Skipped: 0)
Processed: 500 records (Updated: 500, Skipped: 0)
Processed: 600 records (Updated: 600, Skipped: 0)
Processed: 700 records (Updated: 700, Skipped: 0)
Processed: 800 records (Updated: 800, Skipped: 0)
Processed: 900 records (Updated: 900, Skipped: 0)
Processed: 1000 records (Updated: 1000, Skipped: 0)
Processed: 1100 records (Updated: 1100, Skipped: 0)
Processed: 1200 records (Updated: 1200, Skipped: 0)
Processed: 1300 records (Updated: 1300, Skipped: 0)
Processed: 1400 records (Updated: 1400, Skipped: 0)

✓ Update complete!
Total Updated: 1454
Total Skipped: 0
Total Errors: 0


# Update lại poi_type_clean

In [ ]:
# Update poi_type_clean from "Cafe & Bakery" to "Cafe"

try:
    cur.execute(
        '''
        UPDATE "PoiClean"
        SET poi_type_clean = %s
        WHERE poi_type_clean = %s
        ''',
        ("History museum", "Local history museum")
    )

    affected_rows = cur.rowcount
    conn.commit()

    print(f"✓ Updated {affected_rows} records")

except Exception as e:
    conn.rollback()
    print(f"✗ Error updating poi_type_clean: {e}")


✓ Updated 1 records: 'Cafe & Bakery' → 'Cafe'
